# Stage 4 — Corpus Build (Components 1 + 2) — optional Colab path

**This notebook is now optional.** The primary, supported way to run Stage 4
end to end is local — see `fw_audit/stage4_rag/CLAUDE.md` and run
`fw-trace build-corpus` / `fw-trace run` directly against your local Stage 1
rootfs and Stage 2 `stage2/binaries/` directories, no zip/upload round-trip
needed. Use this notebook only if you specifically want to build the corpus
on Colab's free GPU instead of your own machine.

Builds the retrieval corpus for `fw-audit`'s Stage 4 (RAG Sink-to-Source Identifier):
classifies + chunks Stage 1's rootfs and Stage 2's cleaned decompiled C, embeds every
chunk with a Qwen3 embedding model, and indexes it into a persistent ChromaDB collection.

**Output:** a zip containing the Chroma collection + `corpus_report.json`, downloaded at
the end of this notebook. Unzip it locally under `<db_subfolder>/stage4/` — the local
`fw-trace run` driver (Components 3-6) picks up from there.

See `MASTERPLAN_STAGE4.md` (repo root) for the full architecture.

**Before running (on your local machine, not in Colab):** build one upload zip with

```bash
python fw_audit/stage4_rag/colab/package_input.py \
  --rootfs data/db/<stem>/binwalk_1/_input.pkgtb.extracted/squashfs-root \
  --stage2-binaries data/db/<stem>/stage2/binaries \
  --output stage4_colab_input.zip
```

This packages Stage 1's rootfs and Stage 2's `stage2/binaries/` (containing
`<bin_id>/cleaned/whole.c` per binary) into the exact layout Section 2 below expects,
and skips any unreadable filesystem entries a real extracted rootfs tends to contain
(device nodes, unsupported reparse points) rather than aborting the whole zip on the
first one — the same failure `Compress-Archive`/`shutil.make_archive` hit on this
project's own test firmware.

## 1. Install dependencies

Colab-only — these are never added to the `fw-audit` package's own `pyproject.toml`
extras, since this notebook never runs inside the project's local venv.

In [ ]:
!pip install -q chromadb sentence-transformers

## 2. Provide the input folders

Pick ONE of the two cells below — upload a zip, or mount Google Drive — then adjust
the paths in Section 4 to match.

In [ ]:
# Option A — upload a zip containing both folders (rootfs + stage2/binaries), then unzip.
# Skip this cell if you're using Option B (Drive mount) instead.

from google.colab import files

uploaded = files.upload()  # select your zip in the file picker
for name in uploaded:
    if name.endswith(".zip"):
        import zipfile
        with zipfile.ZipFile(name) as zf:
            zf.extractall("/content/stage4_input")
        print(f"Extracted {name} -> /content/stage4_input")

In [ ]:
# Option B — mount Google Drive instead, if the input folders already live there.
# Skip this cell if you used Option A.

from google.colab import drive

drive.mount("/content/drive")
# Then point Section 4's config at paths under /content/drive/MyDrive/...

## 3. Paste `chunk_and_embed.py`

The next cell is the **entire contents** of
`fw_audit/stage4_rag/colab/chunk_and_embed.py` from the repo, pasted verbatim. It is
dependency-light and self-contained by design — no `fw_audit` package install needed.
If the repo file changes, re-copy it here rather than editing this cell independently,
so the two never drift apart.

In [ ]:
# >>> PASTE fw_audit/stage4_rag/colab/chunk_and_embed.py CONTENTS BELOW THIS LINE >>>
#
# (left intentionally empty here — copy the file's contents in before running this
# notebook. Keeping it out-of-line avoids two copies of a ~450-line file silently
# drifting apart inside this repo; the .py file is the source of truth.)
#
# <<< PASTE ABOVE THIS LINE <<<

## 4. Configure and run

Adjust `rootfs_dir` and `stage2_binaries_dir` to match wherever Section 2 put your
input folders. `embedding_model` has a TODO in the script itself — confirm the exact
Qwen3-Embedding checkpoint tag before running (see `MASTERPLAN_STAGE4.md` §14).

In [ ]:
config = Stage4ColabConfig(
    rootfs_dir=Path("/content/stage4_input/squashfs-root"),
    stage2_binaries_dir=Path("/content/stage4_input/stage2_binaries"),
    output_dir=Path("/content/stage4_corpus_build"),
    chunk_words=500,
    embedding_model="Qwen/Qwen3-Embedding-0.6B",
)

zip_path = run(config)

## 5. Download the result

Unzip locally under `<db_subfolder>/stage4/` — you should end up with
`stage4/chroma/` and `stage4/corpus_report.json`.

In [ ]:
from google.colab import files

files.download(str(zip_path))